# CHADSVASC Score extraction for MIMIC-IV-Ext-Cardiology
We use a subset of the dataset, with only AF positive entries cause those are the focus of the Quality Indicator. 

In [ ]:
import re
import pandas as pd

In [ ]:
FILE = r"..\baseline\feather_files\mimiciv_icd_cardiology_AFonly_v3.feather"
print("Loading feather file:", FILE)
df = pd.read_feather(FILE)

In [ ]:
print("Amount of data loaded:", len(df))

## Baseline: REGEX-based retrieval of literal scores

In [ ]:
chads_pattern = r"""
    (?:CHA2?DS?2?|CHADVASC?|CHADS-?2?)   
    [\s\-\_]*                            
    (?:VAS?C?2?)?                           
"""
chads_score_pattern = chads_pattern + r"\s+[^\d]{0,35}?([0-9])"

In [ ]:
def extract_chadsvasc(text):
    match = re.search(chads_score_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        return int(match.group(1))

    return None

In [ ]:
df["chadsvasc_score"] = df["volledig_verslag_report_content"].apply(extract_chadsvasc)
print(df["chadsvasc_score"].value_counts(dropna=False))
num_non_nan = df["chadsvasc_score"].count()
print("Number of non-NaN CHADS-VASc scores:", num_non_nan)

In [ ]:
df["chads_mentioned"] = df["volledig_verslag_report_content"].str.contains(
    chads_pattern, case=False, na=False, regex=True, flags=re.VERBOSE
)
num_mentions = df["chads_mentioned"].sum()
total_docs = len(df)
print(
    f"\nDocuments mentioning CHADS: {num_mentions}/{total_docs} ({num_mentions/total_docs:.2%})"
)


def get_chads_context(text, window=50):
    if not isinstance(text, str):
        return None
    matches = re.finditer(chads_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    contexts = []
    for m in matches:
        start = max(m.start() - window, 0)
        end = min(m.end() + window, len(text))
        contexts.append(text[start:end])
    return contexts if contexts else None


df["chads_context"] = df["volledig_verslag_report_content"].apply(get_chads_context)

# Notes where CHADS-VASc is mentioned but no numeric score extracted
missing_score_mask = df["chads_mentioned"] & df["chadsvasc_score"].isna()
missing_score_notes = df.loc[missing_score_mask, "volledig_verslag_report_content"]

print(
    f"\nNumber of notes mentioning CHADS but no extracted score: {len(missing_score_notes)}"
)

In [ ]:
window = 250

for i, note in enumerate(missing_score_notes.head(20)):
    if not isinstance(note, str):
        continue
    match = re.search(chads_pattern, note, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        start = max(match.start() - window, 0)
        end = min(match.end() + window, len(note))
        context = note[start:end]
        print(f"\nExample {i+1} (context around CHADS-VASc):\n...{context}...")
    else:
        print(f"\nExample {i+1} (no match found)\n")

## Qualitative data

In [ ]:
qualitative_pattern = r"""
    \b(high|low|elevated|raised|moderate|intermediate)\b  
    [\s\-]*                                               
    """ + chads_pattern


def extract_chads_qualitative(text):
    if not isinstance(text, str):
        return None
    match = re.search(qualitative_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        adjective = match.group(1).lower()
        if adjective in ["low"]:
            return 0
        elif adjective in ["moderate", "intermediate"]:
            return 1
        elif adjective in ["high", "elevated", "raised"]:
            return 2
    return None


mask_no_numeric = df["chadsvasc_score"].isna()
df.loc[mask_no_numeric, "chads_qual_class"] = df.loc[
    mask_no_numeric, "volledig_verslag_report_content"
].apply(extract_chads_qualitative)
print("\nExtracted qualitative CHADS classes:")
print(df["chads_qual_class"].value_counts(dropna=False))

##  Greater than / > cases  

In [ ]:
threshold_pattern_greater = chads_pattern + r"""
    [^\d]{0,20}?                        
    (?:>|\bgreater\s+than\b|\bmore\s+than\b|\bat\s+least\b)  
    \s*([0-9])                              
"""

threshold_pattern_smaller = chads_pattern + r"""
    [^\d]{0,20}?                        
    (?:<|\bsmaller\s+than\b|\bless\s+than\b|\bat\s+most\b)  
    \s*([0-9])                              
"""


def extract_threshold_chadsvasc(text, pattern):
    match = re.search(pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        return int(match.group(1))
    return None


### GREATER ###
df["chadsvasc_score_threshold_greater"] = df["volledig_verslag_report_content"].apply(
    extract_threshold_chadsvasc, args=(threshold_pattern_greater,)
)
print(df["chadsvasc_score_threshold_greater"].value_counts(dropna=False))
num_non_nan = df["chadsvasc_score_threshold_greater"].count()
print("Number of CHADS-VASc scores using a greater threshold:", num_non_nan)

# find the entries that have a threshold case and overwrite the found CHADSVASC score from earlier with the old value + 1
mask_threshold = df["chadsvasc_score_threshold_greater"].notna()
df.loc[mask_threshold, "chadsvasc_score"] = (
    df.loc[mask_threshold, "chadsvasc_score"].fillna(0) + 1
)

print(
    df.loc[
        mask_threshold,
        ["hadm_id", "chadsvasc_score", "chadsvasc_score_threshold_greater"],
    ].head()
)

### SMALLER ###
df["chadsvasc_score_threshold_smaller"] = df["volledig_verslag_report_content"].apply(
    extract_threshold_chadsvasc, args=(threshold_pattern_smaller,)
)
print(df["chadsvasc_score_threshold_smaller"].value_counts(dropna=False))
num_non_nan = df["chadsvasc_score_threshold_smaller"].count()
print("Number of CHADS-VASc scores using a smaller threshold:", num_non_nan)

# find the entries that have a threshold case and overwrite the found CHADSVASC score from earlier with the old value - 1
mask_threshold = df["chadsvasc_score_threshold_smaller"].notna()
df.loc[mask_threshold, "chadsvasc_score"] = (
    df.loc[mask_threshold, "chadsvasc_score"].fillna(0) - 1
)

In [ ]:
print(df.columns)

In [ ]:
df = df.drop(
    [
        "chadsvasc_score_threshold_greater",
        "chadsvasc_score_threshold_smaller",
        "chads_context",
    ],
    axis=1,
)

# Component-based reconstruction


|        Component        |            Criteria / Example Keywords            | Points |         Related ICD10 codes        |
|:-----------------------:|:-------------------------------------------------:|:------:|:----------------------------------:|
| C: Congestive HF        | “heart failure”, “CHF”                            | 1      | I50*                               |
| H: Hypertension         | “hypertension”, “HTN”                             | 1      |                                    |
| A2: Age ≥75             | “age 75”, “older adult”, numeric age if available | 2      |                                    |
| D: Diabetes Mellitus    | “diabetes”, “DM”                                  | 1      |                                    |
| S2: Prior stroke/TIA/TE | “stroke”, “TIA”, “thromboembolism”, “CVA”         | 2      | I69* = Oude CVA, Z86.73 = Oude TIA |
| V: Vascular disease     | “MI”, “PAD”, “peripheral artery disease”, “CAD”   | 1      |                                    |
| A: Age 65–74            | “age 65-74”, “65 years”                           | 1      |                                    |
| Sc: Female sex          | Usually documented as “female”                    | 1      |                                    |

**Problem with this approach**  
Earlier we merged the ICD9 and ICD10 cases of AF, but we only kept the ICD10 codes.  
So only a small percentage will probably have these codes, there are false negatives who still have the old ICD9 variant. 


## Using the available ICD10 codes
Maybe we can find other “>2”/critical patients, just by using the ICD10 codes.

In [ ]:
# Congestive Heart Failure columns
chf_cols = [
    "diagnose_i501",
    "diagnose_i502",
    "diagnose_i503",
    "diagnose_i504",
    "diagnose_i508",
    "diagnose_i509",
]
mask_chf = df[chf_cols].any(axis=1)

# Stroke / TIA columns
mask_stroke = (df["diagnose_i69"] == 1) | (df["diagnose_z8673"] == 1)

mask_missing_score = df["chadsvasc_score"].isna()

mask = mask_chf & mask_stroke & mask_missing_score
df_chf_stroke_missing = df[mask]

print(f"Number of notes matching criteria: {len(df_chf_stroke_missing)}")

These are patients whose ChadsVasc score would have been at least 3, if reported.   
We can easily check if there are cases where the ChadsVasc score is wrongly assigned in the discharge summary. 

In [ ]:
low_score = (
    (df["chadsvasc_score"] == 1)
    | (df["chadsvasc_score"] == 2)
    | (df["chadsvasc_score"] == 0)
)
mask = mask_chf & mask_stroke & low_score
df_chf_stroke_missing = df[mask]

print(
    f"Number of notes with a wrongly assigned ChadsVasc-score: {len(df_chf_stroke_missing)}"
)

print()

We can have a closer look at the area around the word "CHADSVAsc".

In [ ]:
window = 250

for i, row in enumerate(df_chf_stroke_missing.head(20).itertuples()):
    note = row.volledig_verslag_report_content
    score = row.chadsvasc_score

    if not isinstance(note, str):
        continue

    match = re.search(chads_pattern, note, flags=re.IGNORECASE | re.VERBOSE)

    if match:
        start = max(match.start() - window, 0)
        end = min(match.end() + window, len(note))
        context = note[start:end]
        print(
            f"\nExample {i+1} (CHADS-VASc score: {score})\nContext around mention:\n...{context}..."
        )
    else:
        print(f"\nExample {i+1} (CHADS-VASc score: {score})\nNo match found\n")

## Using REGEX for the other criteria

### Extracting the age from unstructured notes

In [ ]:
def extract_age(text):
    if not isinstance(text, str):
        return None

    pattern = r"""
        \b(?:age\s*[:\-]?\s*|           # 'age: 68' or 'age 68'
           (\d{1,3})\s*[-]?\s*yo|       # '68 yo'
           (\d{1,3})\s*y/o|             # '68 y/o'
           (\d{1,3})\s*years?\sold|     # '68 years old'
           (\d{1,3})\s*-\s*year\s*old|  # '68-year-old'
           (\d{1,3})\s*year\s*old)      # '68 year old'
    """

    match = re.search(pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        for g in match.groups():
            if g:
                return int(g)
    return None


df["age"] = df["volledig_verslag_report_content"].apply(extract_age)
print(df["age"].value_counts(dropna=False))
num_non_nan = df["age"].count()
print("Number of non-NaN age extractions:", num_non_nan)

All ages are masked.

### Extracting sex from unstructured notes

In [ ]:
import re


def extract_sex(text):
    if not isinstance(text, str):
        return None

    pattern = r"""
        \b(?:sex|gender)\s*[:\-]?\s*  # optional "Sex:" or "Gender:"
        (male|female|m|f)\b           # capture male/female/M/F
        |
        \b(male|female)\b              # standalone 'male' or 'female'
    """

    match = re.search(pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        sex = next(g for g in match.groups() if g)
        sex = sex.lower()
        if sex.startswith("m"):
            return "M"
        elif sex.startswith("f"):
            return "F"
    return None


df["sex"] = df["volledig_verslag_report_content"].apply(extract_sex)

print(df["sex"].value_counts(dropna=False))

### Extracting hypertension from unstructured notes ###
Approach: first exlude the family history, cause this could infere with the result. To do this, we need to be able to split the note into segments. We can use the same strategy used in "Towards Semi-Structured Automatic ICD Coding via Tree-based Contrastive Learning", where an algorithm is developed for this purpose. It is based on document frequency-inverse average phrase frequency, with the thought that a title appear a lot of times across notes, but only once within a note.   

The code is available, so after executing, we get the following titles:

```
Top titles:
discharge date
___ admission date
___ date of birth
sex
___ family history
unit no
discharge instructions
name
chief complaint
discharge disposition
discharge condition
history of present illness
discharge medications
pertinent results
medications on admission
past medical history
medicine allergies
brief hospital course
followup instructions
facility
social history
activity status
mental status
```

We see many meta-data related titles, so we first try to remove those.

In [ ]:
for i, text in enumerate(df["volledig_verslag_report_content"].head(2)):
    if not isinstance(text, str):
        continue
    tokens = text.split()
    first_700 = tokens[:700]
    snippet = " ".join(first_700)
    print(f"\nNote {i+1} (first 700 tokens):\n{snippet}\n")

In [ ]:
import re


def remove_metadata(note: str) -> str:
    metadata_keys = [
        r"Name:",
        r"Unit No:",
        r"Admission Date:",
        r"Discharge Date:",
        r"Date of Birth:",
        r"Sex:",
        r"Service:",
        r"Allergies:",
        r"Attending:",
    ]

    pattern = re.compile(
        r"^(?:" + "|".join(metadata_keys) + r")\s*:?.*(?:\n[ \t]+.*)*",
        flags=re.MULTILINE | re.IGNORECASE,
    )

    cleaned_note = re.sub(pattern, "", note)

    cleaned_note = re.sub(r"\n\s*\n+", "\n\n", cleaned_note)

    return cleaned_note.strip()

In [ ]:
df["cleaned_report"] = df["volledig_verslag_report_content"].apply(remove_metadata)

df.to_feather(r"..\baseline\feather_files\cleaned_notes_AF_only.feather")

I reran the title extraction procedure and get the following, more relevant results:

```
major surgical or invasive procedure
discharge condition
followup instructions
discharge medications
pertinent results
discharge instructions
history of present illness
social history
___ family history
discharge disposition
chief complaint
medications on admission
discharge diagnosis
brief hospital course
past medical history
facility
physical exam
activity status
mental status
level of consciousness
alert and interactive
heent
clear and coherent
```

In [ ]:
import pandas as pd
import re
import pprint

section_titles = [
    "major surgical or invasive procedure",
    "discharge condition",
    "followup instructions",
    "discharge medications",
    "pertinent results",
    "discharge instructions",
    "history of present illness",
    "social history",
    "family history",
    "discharge disposition",
    "chief complaint",
    "medications on admission",
    "discharge diagnosis",
    "brief hospital course",
    "past medical history",
    "facility",
    "physical exam",
    "activity status",
    "mental status",
    "level of consciousness",
    # "alert and interactive", # was mainly used in sentences as 'level of consciousness': 'alert and interactive.' eg
    # "heent",
    # "clear and coherent" # was mainly used in sentences as 'mental status': 'Clear and coherent.' eg
]

section_pattern = re.compile(
    r"(?i)^(" + "|".join([re.escape(title) for title in section_titles]) + r")\s*[:\n]",
    re.MULTILINE,
)


def split_sections(text):
    sections = {}
    matches = list(section_pattern.finditer(text))
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_name = match.group(1).lower()
        sections[section_name] = text[start:end].strip()
    return sections


df["sections"] = df["cleaned_report"].apply(split_sections)
print(df["sections"].head())
print(df.columns)


pprint.pprint(df["sections"].iloc[0])


df.to_feather(r"..\baseline\feather_files\cleaned_AF_notes_with_sections.feather")

In [ ]:
relevant_sections = [
    "past medical history",
    "history of present illness",
    "brief hospital course",
    "discharge diagnosis",
    "medications on admission",
    "chief complaint",
]

## Hypertension

In [ ]:
import re

# Common hypertension mentions
HYPERTENSION_PATTERNS = [
    r"\bhypertension\b",
    r"\bhigh blood pressure\b",
    r"\bhtn\b",
    r"\bh/o hypertension\b",  # history of
]

# Negations
NEGATION_PATTERNS = [
    r"\bno hypertension\b",
    r"\bdenies hypertension\b",
    r"\bwithout hypertension\b",
    r"\bnegative for hypertension\b",
]

In [ ]:
def detect_patterns(sections, positive_patterns, negative_patterns):
    text = " ".join(sections.get(sec, "").lower() for sec in relevant_sections)
    if not text.strip():
        return None

    if any(re.search(p, text) for p in negative_patterns):
        return 0
    if any(re.search(p, text) for p in positive_patterns):
        return 1
    return None

In [ ]:
df["hypertension"] = df["sections"].apply(
    detect_patterns, args=(HYPERTENSION_PATTERNS, NEGATION_PATTERNS)
)

print(df["hypertension"].value_counts(dropna=False))
num_non_nan = df["hypertension"].count()
print("Number of non-NaN hypertension scores:", num_non_nan)

## Diabetes extraction

In [ ]:
# Common diabetes mentions
DIABETES_PATTERNS = [
    r"\bdiabetes\b",
    r"\bdiabetes mellitus\b",
    r"\btype\s*[12]\s*diabetes\b",
    r"\btype\s*[12]\s*dm\b",
    r"\bdm\b",  # abbreviation for diabetes mellitus
    r"\bnon[-\s]*insulin[-\s]*dependent\b",
    r"\binsulin[-\s]*dependent\b",
    r"\bdiabetic\b",  # e.g., "diabetic neuropathy"
]

# Negations / exclusions
DIABETES_NEGATION_PATTERNS = [
    r"\bno diabetes\b",
    r"\bdenies diabetes\b",
    r"\bwithout diabetes\b",
    r"\bnegative for diabetes\b",
    r"\bno history of diabetes\b",
    r"\bnon[-\s]*diabetic\b",
]

In [ ]:
df["diabetes"] = df["sections"].apply(
    detect_patterns, args=(DIABETES_PATTERNS, DIABETES_NEGATION_PATTERNS)
)
print(df["diabetes"].value_counts(dropna=False))
num_non_nan = df["diabetes"].count()
print("Number of non-NaN diabetes scores:", num_non_nan)

## Extract cardivascular disease

In [ ]:
# Common vascular disease mentions
VASCULAR_DISEASE_PATTERNS = [
    r"\bvascular disease\b",
    r"\bperipheral (artery|arterial|vascular) disease\b",
    r"\bpad\b",  # abbreviation for peripheral artery disease
    r"\bperipheral atherosclerosis\b",
    r"\bcoronary artery disease\b",
    r"\bcad\b",
    r"\bischemic heart disease\b",
    r"\bmyocardial infarction\b",
    r"\bmi\b",
    r"\baortic plaque\b",
    r"\batherosclerosis\b",
    r"\bangioplasty\b",
    r"\bbypass surgery\b",
    r"\bcarotid stenosis\b",
]

# Negations and exclusions
VASCULAR_DISEASE_NEGATION_PATTERNS = [
    r"\bno vascular disease\b",
    r"\bdenies vascular disease\b",
    r"\bwithout vascular disease\b",
    r"\bno evidence of vascular disease\b",
    r"\bno (history|hx) of vascular disease\b",
    r"\bno cad\b",
    r"\bdenies cad\b",
    r"\bno mi\b",
    r"\bno history of (mi|myocardial infarction)\b",
    r"\bno (pad|peripheral artery disease)\b",
    r"\bnon[-\s]*ischemic\b",
]

In [ ]:
df["cardivascular"] = df["sections"].apply(
    detect_patterns,
    args=(VASCULAR_DISEASE_PATTERNS, VASCULAR_DISEASE_NEGATION_PATTERNS),
)
print(df["cardivascular"].value_counts(dropna=False))
num_non_nan = df["cardivascular"].count()
print("Number of non-NaN cardivascular scores:", num_non_nan)

## Inferred CHADSVASC score

In [ ]:
chf_cols = [
    "diagnose_i501",
    "diagnose_i502",
    "diagnose_i503",
    "diagnose_i504",
    "diagnose_i508",
    "diagnose_i509",
]
mask_chf = df[chf_cols].any(axis=1).astype(int)

mask_stroke = ((df["diagnose_i69"] == 1) | (df["diagnose_z8673"] == 1)).astype(int)

sex_points = (df["sex"].fillna("M").str.upper() == "F").astype(int)

df["inferred_chadsvasc"] = (
    mask_chf
    + df["hypertension"].fillna(0).astype(int)
    + df["diabetes"].fillna(0).astype(int)
    + mask_stroke * 2
    + df["cardivascular"].fillna(0).astype(int)
    + sex_points
)

print(df["inferred_chadsvasc"].value_counts(dropna=False))

In [ ]:
df["chadsvasc_score"] = pd.to_numeric(df["chadsvasc_score"], errors="coerce")

df_compare = df.dropna(subset=["chadsvasc_score"]).copy()

df_compare["score_diff"] = (
    df_compare["inferred_chadsvasc"] - df_compare["chadsvasc_score"]
)

print(df_compare[["inferred_chadsvasc", "chadsvasc_score", "score_diff"]].describe())

print("Exact match rate:", (df_compare["score_diff"] == 0).mean())
print("Difference counts:")
print(df_compare["score_diff"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_compare = df.dropna(subset=["chadsvasc_score"]).copy()
df_compare["score_diff"] = (
    df_compare["inferred_chadsvasc"] - df_compare["chadsvasc_score"]
)

plt.figure(figsize=(8, 5))
sns.countplot(x="score_diff", data=df_compare, color="skyblue")
plt.title("Distribution of Difference: Inferred - Documented CHADS-VASc")
plt.xlabel("Score Difference")
plt.ylabel("Count")
plt.show()